# Cleaner Runtime StateGraph Real Test

使用 `notebooks._helpers` 中的默认 MinIO `sample_1000` 数据集，验证 TOML 配方、dry-run、compile、运行时进度、单算子预览、导出和 cleanup。

## 0. Bootstrap repo root

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root")
for import_root in [repo_root / "src", repo_root]:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

repo_root

PosixPath('/home/wuchaoli/codespace/ImageGallery/.worktrees/cleaner-runtime-stategraph')

## 1. Imports and output directories

In [2]:
import re
import zipfile

import pandas as pd
from image_gallery.cleaning import BasicCleaner
from notebooks._helpers.cleaning_configs import get_cleaning_v3_non_semantic_all_operator_configs
from notebooks._helpers.datasets import load_default_minio_sample_1000_dataset, load_default_minio_sample_1000_frame
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

pd.set_option("display.max_columns", 80)

RUN_ROOT = get_notebook_library_root("cleaner_runtime_stategraph_real_test")
RECIPE_DIR = reset_output_dir(RUN_ROOT / "recipe")
EXPORT_DIR = reset_output_dir(RUN_ROOT / "exports")
PREVIEW_DIR = reset_output_dir(RUN_ROOT / "previews")

print("recipe_dir:", RECIPE_DIR)
print("export_dir:", EXPORT_DIR)
print("preview_dir:", PREVIEW_DIR)

recipe_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaner-runtime-stategraph/datasets/tests/cleaner_runtime_stategraph_real_test/recipe
export_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaner-runtime-stategraph/datasets/tests/cleaner_runtime_stategraph_real_test/exports
preview_dir: /home/wuchaoli/codespace/ImageGallery/.worktrees/cleaner-runtime-stategraph/datasets/tests/cleaner_runtime_stategraph_real_test/previews


## 2. Load sample_1000 dataset

In [3]:
raw_frame = load_default_minio_sample_1000_frame()
dataset = load_default_minio_sample_1000_dataset()

required_columns = {"image_id", "image_uri"}
missing_columns = sorted(required_columns.difference(raw_frame.columns))
if missing_columns:
    raise AssertionError(f"missing raw dataset columns: {missing_columns}")
if len(raw_frame) != 1000:
    raise AssertionError(f"expected sample_1000 to contain 1000 rows, got {len(raw_frame)}")

raw_frame[["image_id", "image_uri"]].head()

,image_id,image_uri
0,001ab42f-e567-4912-bcb6-ea5bcdbc8280,s3://test/images/raw/2026-07-06/shard_002/001a...
1,00a6d3a7-c561-493c-b0d6-978c876f6b94,s3://test/images/raw/2026-07-06/shard_002/00a6...
2,016cf5f4-4d51-456d-8acb-805874c0e259,s3://test/images/raw/2026-07-06/shard_002/016c...
3,01fcce40-1bbd-4395-bc31-c89accd1e1d6,s3://test/images/raw/2026-07-06/shard_001/01fc...
4,020706b0-26eb-48ec-ac98-7ee816cea90c,s3://test/images/raw/2026-07-06/shard_001/0207...


## 3. Write and load YAML recipe

In [4]:
operator_configs = get_cleaning_v3_non_semantic_all_operator_configs()
operator_names = [next(iter(item)) for item in operator_configs]
recipe_path = RECIPE_DIR / "cleaner_runtime_non_semantic_all.toml"

def toml_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return f'"{value}"'
    return repr(value)

recipe_lines = ["[cleaner]", "operators = [" + ", ".join(f'\"{name}\"' for name in operator_names) + "]", ""]
for item in operator_configs:
    operator_name, config = next(iter(item.items()))
    recipe_lines.extend(["[[operator]]", f'name = "{operator_name}"'])
    recipe_lines.extend(f"{key} = {toml_value(value)}" for key, value in config.items())
    recipe_lines.append("")
recipe_path.write_text("\n".join(recipe_lines), encoding="utf-8")

cleaner = BasicCleaner.from_toml(recipe_path)
print("operator_count:", len(operator_names))
recipe_path

operator_count: 16


PosixPath('/home/wuchaoli/codespace/ImageGallery/.worktrees/cleaner-runtime-stategraph/datasets/tests/cleaner_runtime_stategraph_real_test/recipe/cleaner_runtime_non_semantic_all.toml')

## 4. Compile

In [5]:
execution = cleaner.compile()
plan_frame = execution.plan()

expected_evaluation_nodes = {f"evaluation.{operator_name}" for operator_name in operator_names}
missing_nodes = sorted(expected_evaluation_nodes.difference(set(plan_frame["node_id"])))
if missing_nodes:
    raise AssertionError(f"missing evaluation nodes: {missing_nodes}")

dry_run = execution.dry_run(dataset)
if dry_run.errors:
    raise AssertionError(f"dry-run errors: {dry_run.errors}")

plan_frame

,node_id,node_type,operator_name,computer_name,stage_name,execution_mode,required_parameters,produced_parameters,upstream_node_ids,config_hash,policy_hash,checkpoint_strategy,required_artifacts,produced_artifacts,required_relations,produced_relations,artifact_contract,cache_policy
0,parameter.image_border_computer,parameter,None,image_border_computer,parameter,per_image,[],"[border_padding_color, border_padding_ratio, b...",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
1,parameter.image_format_detail_computer,parameter,None,image_format_detail_computer,parameter,per_image,[],"[animated, exif_orientation, frame_count, orie...",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
2,parameter.image_hash_computer,parameter,None,image_hash_computer,parameter,per_image,[],[content_hash],[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
3,parameter.image_metadata_computer,parameter,None,image_metadata_computer,parameter,per_image,[],"[decode_error, decode_ok, file_size, format, h...",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
4,parameter.image_perceptual_hash_computer,parameter,None,image_perceptual_hash_computer,parameter,per_image,[],[phash],[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
5,parameter.image_quality_computer,parameter,None,image_quality_computer,parameter,per_image,[],"[blank_score, blur_score, brightness_score, co...",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
6,parameter.image_quality_detail_computer,parameter,None,image_quality_detail_computer,parameter,per_image,[],"[bright_pixel_ratio, clipped_pixel_ratio, dark...",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
7,parameter.table_derived_computer,parameter,None,table_derived_computer,parameter,table,[],"[aspect_ratio, megapixels]",[],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,batch,[],[],[],[],none,run
8,parameter.duplicate_group_computer,parameter,None,duplicate_group_computer,parameter,dataset_aggregate,[content_hash],"[exact_duplicate_count, exact_duplicate_group_id]",[parameter.image_hash_computer],default,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,whole_node,[],[],[],[],none,run
9,parameter.perceptual_duplicate_group_computer,parameter,None,perceptual_duplicate_group_computer,parameter,dataset_aggregate,[phash],"[perceptual_duplicate_count, perceptual_duplic...",[parameter.image_perceptual_hash_computer],2e7de0f52cfced4e046ffac5c52563420f2f8b227604e2...,e0e2ef5934fdc3a93f37fe8506f027cf89608d84a0da11...,whole_node,[],[],[],[],none,run


## 5. Run with notebook progress

In [6]:
result = execution.run(
    dataset,
    sample=1000,
    label="cleaner-runtime-stategraph-real-test",
    tags=["real", "sample_1000", "stategraph"],
    progress="auto",
)

if result.status() != "completed":
    raise AssertionError(f"unexpected result status: {result.status()}")

state_frame = result.state()
if set(operator_names).difference(set(state_frame["operator_name"])):
    raise AssertionError("operator state does not cover all recipe operators")

state_frame

[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] run_started runtime - runtime started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started parameter.stage - parameter stage started


[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed parameter.stage - parameter stage completed


[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.format.decode_check - evaluation.format.decode_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.format.decode_check - evaluation.format.decode_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.size.dimension_check - evaluation.size.dimension_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.size.dimension_check - evaluation.size.dimension_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.size.aspect_ratio_check - evaluation.size.aspect_ratio_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.size.aspect_ratio_check - evaluation.size.aspect_ratio_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.size.megapixel_check - evaluation.size.megapixel_check started


[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.size.megapixel_check - evaluation.size.megapixel_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.quality.blur_check - evaluation.quality.blur_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.quality.blur_check - evaluation.quality.blur_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.quality.brightness_check - evaluation.quality.brightness_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.quality.brightness_check - evaluation.quality.brightness_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.quality.contrast_check - evaluation.quality.contrast_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.quality.contrast_check - evaluation.quality.contrast_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started ev

[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.duplicate.perceptual_duplicate_check - evaluation.duplicate.perceptual_duplicate_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.quality.exposure_check - evaluation.quality.exposure_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.quality.exposure_check - evaluation.quality.exposure_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.content.border_padding_check - evaluation.content.border_padding_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.content.border_padding_check - evaluation.content.border_padding_check completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started evaluation.quality.noise_check - evaluation.quality.noise_check started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed evaluation.quality.noise_check - evaluation.quality.noise_check completed
[clean

[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_started merge.final_action - merge.final_action started
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] node_completed merge.final_action - merge.final_action completed
[cleaner:05b1af621f094fc4aa7b91a4c872dfd8] run_completed runtime - run completed


,operator_name,status,processed_count,skipped_count,failed_count,message
0,format.decode_check,completed,1000,0,0,None
1,size.dimension_check,completed,1000,0,0,None
2,size.aspect_ratio_check,completed,1000,0,0,None
3,size.megapixel_check,completed,1000,0,0,None
4,quality.blur_check,completed,1000,0,0,None
5,quality.brightness_check,completed,1000,0,0,None
6,quality.contrast_check,completed,1000,0,0,None
7,content.blank_image_check,completed,1000,0,0,None
8,duplicate.exact_duplicate_check,completed,1000,0,0,None
9,duplicate.perceptual_duplicate_check,completed,1000,0,0,None


## 6. Export one HTML preview per operator

In [7]:
preview_rows = []
for operator_name in operator_names:
    operator_frame = result.result(operator_name)
    action_columns = [column for column in operator_frame.columns if column.endswith("_action")]
    if len(action_columns) != 1:
        raise AssertionError(f"expected one action column for {operator_name}, got {action_columns}")
    action_column = action_columns[0]
    drop_count = int((operator_frame[action_column] == "drop").sum())
    review_count = int((operator_frame[action_column] == "review").sum())
    preview_count = drop_count + review_count
    preview_path = result.preview_html(
        PREVIEW_DIR / f"{operator_name.replace('.', '__')}.html",
        operator_name=operator_name,
        actions=["drop", "review"],
        max_rows=dataset.count(),
        max_items_per_group=dataset.count(),
    )
    preview_html = preview_path.read_text(encoding="utf-8")
    rows_match = re.search(r"<strong>rows</strong>: (\d+)", preview_html)
    if rows_match is None or int(rows_match.group(1)) != preview_count:
        actual_count = None if rows_match is None else int(rows_match.group(1))
        raise AssertionError(
            f"HTML preview row count mismatch for {operator_name}: "
            f"expected {preview_count}, got {actual_count}"
        )
    preview_rows.append(
        {
            "operator_name": operator_name,
            "drop_count": drop_count,
            "review_count": review_count,
            "preview_count": preview_count,
            "preview_path": str(preview_path),
        }
    )

preview_paths = [Path(row["preview_path"]) for row in preview_rows]
missing_preview_paths = [path for path in preview_paths if not path.exists()]
if missing_preview_paths:
    raise AssertionError(f"missing preview files: {missing_preview_paths}")

preview_summary = pd.DataFrame(preview_rows)
preview_summary

,operator_name,drop_count,review_count,preview_count,preview_path
0,format.decode_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
1,size.dimension_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
2,size.aspect_ratio_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
3,size.megapixel_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
4,quality.blur_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
5,quality.brightness_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
6,quality.contrast_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
7,content.blank_image_check,0,0,0,/home/wuchaoli/codespace/ImageGallery/.worktre...
8,duplicate.exact_duplicate_check,2,0,2,/home/wuchaoli/codespace/ImageGallery/.worktre...
9,duplicate.perceptual_duplicate_check,207,0,207,/home/wuchaoli/codespace/ImageGallery/.worktre...


## 7. Export tables, datasets, relations, and debug bundle

In [8]:
parameter_table_path = result.export_table("parameter", EXPORT_DIR / "parameter_table.parquet")
evaluation_table_path = result.export_table("evaluation", EXPORT_DIR / "evaluation_table.parquet")
execution_plan_path = result.export_manifest("execution_plan", EXPORT_DIR / "execution_plan.json")
artifacts_manifest_path = result.export_manifest("artifacts", EXPORT_DIR / "artifacts_manifest.json")
perceptual_relation_path = result.export_relations("perceptual_duplicate_pairs", EXPORT_DIR / "perceptual_duplicate_pairs.parquet")
debug_bundle_path = result.export_debug_bundle(EXPORT_DIR / "debug_bundle.zip")

exported_datasets = {
    kind: result.export(kind, EXPORT_DIR / f"{kind}.parquet")
    for kind in ["full", "clean", "review", "dropped"]
}

parameter_table = pd.read_parquet(parameter_table_path)
evaluation_table = pd.read_parquet(evaluation_table_path)
if len(parameter_table) != 1000 or len(evaluation_table) != 1000:
    raise AssertionError("exported runtime tables must preserve sample_1000 row count")
if exported_datasets["full"].count() != 1000:
    raise AssertionError("full export must preserve sample_1000 row count")
if not debug_bundle_path.exists():
    raise AssertionError("debug bundle was not exported")
with zipfile.ZipFile(debug_bundle_path) as archive:
    archive_names = set(archive.namelist())
if "tables/parameter_table.parquet" not in archive_names:
    raise AssertionError("debug bundle missing parameter table")

pd.DataFrame(
    {
        "artifact": [
            "parameter",
            "evaluation",
            "execution_plan",
            "artifacts_manifest",
            "perceptual_relations",
            "debug_bundle",
            "full",
            "clean",
            "review",
            "dropped",
        ],
        "path": [
            str(parameter_table_path),
            str(evaluation_table_path),
            str(execution_plan_path),
            str(artifacts_manifest_path),
            str(perceptual_relation_path),
            str(debug_bundle_path),
            str(EXPORT_DIR / "full.parquet"),
            str(EXPORT_DIR / "clean.parquet"),
            str(EXPORT_DIR / "review.parquet"),
            str(EXPORT_DIR / "dropped.parquet"),
        ],
    }
)

,artifact,path
0,parameter,/home/wuchaoli/codespace/ImageGallery/.worktre...
1,evaluation,/home/wuchaoli/codespace/ImageGallery/.worktre...
2,execution_plan,/home/wuchaoli/codespace/ImageGallery/.worktre...
3,artifacts_manifest,/home/wuchaoli/codespace/ImageGallery/.worktre...
4,perceptual_relations,/home/wuchaoli/codespace/ImageGallery/.worktre...
5,debug_bundle,/home/wuchaoli/codespace/ImageGallery/.worktre...
6,full,/home/wuchaoli/codespace/ImageGallery/.worktre...
7,clean,/home/wuchaoli/codespace/ImageGallery/.worktre...
8,review,/home/wuchaoli/codespace/ImageGallery/.worktre...
9,dropped,/home/wuchaoli/codespace/ImageGallery/.worktre...


## 8. Cleanup runtime data

In [9]:
result.cleanup()

if not EXPORT_DIR.exists():
    raise AssertionError("cleanup must not remove exported tables")
if not PREVIEW_DIR.exists():
    raise AssertionError("cleanup must not remove preview HTML files")

print("PASS: cleaner runtime stategraph real notebook smoke completed")

PASS: cleaner runtime stategraph real notebook smoke completed
